In [1]:
pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 27.4 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 28.3 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [numpy]  WARNING: The scripts f2py and numpy-config are installed in '/usr/local/python/3.12.1/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [4]:
import pandas as pd
df = pd.read_excel('online_retail_II.xlsx')
df.head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


In [5]:
missing_count = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_table = pd.DataFrame({
    'count of missing values' : missing_count , 
    'percentage of missing value' : missing_percentage
})
missing_table = missing_table[missing_table['count of missing values'] > 0].sort_values(by='percentage of missing value', ascending=False)
print(missing_table)

             count of missing values  percentage of missing value
Customer ID                   107927                    20.539488
Description                     2928                     0.557225


In [33]:
import sqlite3
import pandas as pd

df['amount'] = df['Quantity'] * df['Price']

conn = sqlite3.connect('retail_database.db')
df.to_sql('sales', conn, if_exists='replace', index=False)
print(" data added to sqlite ")

 data added to sqlite 


In [34]:
# sales monthly based on region 
query = """SELECT 
    strftime('%Y-%m', InvoiceDate) AS month,
    Country,
    SUM(amount) AS total_sales
FROM sales
GROUP BY strftime('%Y-%m', InvoiceDate), Country
ORDER BY month ASC, total_sales DESC; """
mounthly_sales = pd.read_sql_query(query,conn)
print(mounthly_sales)

       month          Country  total_sales
0    2009-12   United Kingdom    727655.96
1    2009-12             EIRE     19127.94
2    2009-12      Netherlands     15204.73
3    2009-12          Germany      9782.62
4    2009-12            Spain      7918.83
..       ...              ...          ...
293  2010-12  Channel Islands       363.53
294  2010-12          Belgium       346.10
295  2010-12      Switzerland       303.40
296  2010-12           Poland       248.16
297  2010-12      Netherlands       192.60

[298 rows x 3 columns]


In [35]:
# 5 TOP product
query = """SELECT 
    Description,
    SUM(Quantity) AS total_quantity
FROM sales
WHERE Description IS NOT NULL
GROUP BY Description
ORDER BY total_quantity DESC
LIMIT 5;"""
total_quantity = pd.read_sql_query(query,conn)
print('total quantity : ')
print(total_quantity)

total quantity : 
                          Description  total_quantity
0  WHITE HANGING HEART T-LIGHT HOLDER           57733
1   WORLD WAR 2 GLIDERS ASSTD DESIGNS           54698
2                 BROCADE RING PURSE            47647
3    PACK OF 72 RETRO SPOT CAKE CASES           46106
4       ASSORTED COLOUR BIRD ORNAMENT           44925


In [36]:
# 5 top profitable product
query = """SELECT 
    Description,
    SUM(amount) AS total_revenue
FROM sales
WHERE Description IS NOT NULL
GROUP BY Description
ORDER BY total_revenue DESC
LIMIT 5;"""
total_revenue = pd.read_sql_query(query,conn)
print(total_revenue)

                          Description  total_revenue
0            REGENCY CAKESTAND 3 TIER      163051.46
1  WHITE HANGING HEART T-LIGHT HOLDER      157865.43
2                      DOTCOM POSTAGE      116401.99
3       ASSORTED COLOUR BIRD ORNAMENT       72454.12
4     PAPER CHAIN KIT 50'S CHRISTMAS        57870.20


In [38]:
# top customers
query = """SELECT 
    `Customer ID`,
    SUM(amount) AS total_spent,
    COUNT(DISTINCT Invoice) AS total_orders
FROM sales
WHERE `Customer ID` IS NOT NULL
GROUP BY `Customer ID`
ORDER BY total_spent DESC
LIMIT 5;"""
top_customers = pd.read_sql_query(query,conn)
print(top_customers)

   Customer ID  total_spent  total_orders
0      18102.0    341776.73            95
1      14646.0    243853.05            87
2      14156.0    183180.55           138
3      14911.0    137675.91           270
4      13694.0    128172.42           105


In [39]:
# status of sales in different hours 
query = """SELECT 
    strftime('%H:00', InvoiceDate) AS hour_of_day,
    COUNT(DISTINCT Invoice) AS number_of_invoices,
    SUM(amount) AS total_sales
FROM sales
GROUP BY strftime('%H', InvoiceDate)
ORDER BY total_sales DESC;"""
status_sales = pd.read_sql_query(query,conn)
print(status_sales)

   hour_of_day  number_of_invoices  total_sales
0        12:00                4290  1365831.561
1        13:00                3936  1213628.634
2        11:00                3560  1204162.923
3        14:00                3516  1155534.430
4        10:00                3180  1127537.702
5        15:00                3299   979084.932
6        16:00                2295   805010.331
7        09:00                1767   783137.620
8        17:00                1388   400726.571
9        08:00                 548   242582.360
10       18:00                 513   124735.330
11       19:00                 367    74480.720
12       07:00                  81    42931.900
13       20:00                  75    20104.570
14       21:00                   1       -4.950


In [40]:
conn.close()

In [41]:
!pip install streamlit plotly pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 82.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.6/797.6 kB 32.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 85.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 85.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 84.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 50.0 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [streamlit]20 [streamlit]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [43]:
! pip install panel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.3/30.3 MB 80.6 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 103.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 53.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.7/806.7 kB 34.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [panel]m14/15 [panel]material-ui]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [44]:
import plotly.express as px
import panel as pn
pn.extension('plotly')

/tmp/ipykernel_51987/1526321449.py:3: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension('plotly')


In [45]:
conn = sqlite3.connect('retail_database.db')
query_kpi = "SELECT SUM(amount) AS total FROM sales;"
total_revenue = pd.read_sql_query(query_kpi, conn)['total'].iloc[0]

In [46]:
query_trend = """
SELECT 
    strftime('%Y-%m', InvoiceDate) AS Month,
    SUM(amount) AS TotalSales
FROM sales
GROUP BY Month
ORDER BY Month ASC;
"""
df_trend = pd.read_sql_query(query_trend, conn)

In [47]:
query_top_products = """
SELECT 
    Description,
    SUM(amount) AS TotalRevenue
FROM sales
WHERE Description IS NOT NULL
GROUP BY Description
ORDER BY TotalRevenue DESC
LIMIT 5;
"""
df_top_products = pd.read_sql_query(query_top_products, conn)

conn.close()

In [48]:
kpi_card = pn.indicators.Number(
    name='Total revenue of sales from real data', 
    value=total_revenue, 
    format='${value:,.2f}',
    font_size='26pt',
    title_size='12pt'
)

In [49]:
fig_trend = px.line(
    df_trend, x='Month', y='TotalSales', 
    title='sale of company monthly', markers=True
)
pane_trend = pn.pane.Plotly(fig_trend)

In [50]:
fig_products = px.bar(
    df_top_products, x='TotalRevenue', y='Description', 
    orientation='h', title='5 Profitable best-selling product',
    color='TotalRevenue', color_continuous_scale='Blues'
)

In [51]:
fig_products.update_layout(yaxis={'categoryorder':'total ascending'}) 
pane_products = pn.pane.Plotly(fig_products)


In [52]:
dashboard = pn.Column(
    pn.pane.Markdown("Ultimate online sales management dashboard", styles={'font-family': 'tahoma'}),
    kpi_card,
    pn.Spacer(height=15),
    pn.Row(pane_trend, pane_products) 
)

In [53]:
dashboard

Column
    [0] Markdown(str, styles={'font-family': 'tahoma'})
    [1] Number(font_size='26pt', format='${value:,.2f}', label='Total revenue o..., name='Total revenue o..., title_size='12pt', value=np.float64(9539484.634))
    [2] Spacer(height=15)
    [3] Row
        [0] Plotly(Figure)
        [1] Plotly(Figure)